<a href="https://colab.research.google.com/github/abhuvan345/6thSem-ML-Lab/blob/main/1BM24CS403_Lab_7_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
from collections import Counter
from sklearn.datasets import load_iris

# ---------------------------
# Load Dataset (No CSV needed)
# ---------------------------
def load_data():
    iris = load_iris()
    dataset = [(list(x), str(y)) for x, y in zip(iris.data, iris.target)]
    return dataset


# ---------------------------
# Split Dataset
# ---------------------------
def train_test_split(dataset, test_size=0.2):
    random.shuffle(dataset)
    split = int(len(dataset) * (1 - test_size))
    return dataset[:split], dataset[split:]


# ---------------------------
# Gini Index
# ---------------------------
def gini(groups, classes):
    total = sum(len(group) for group in groups)
    gini_score = 0.0

    for group in groups:
        size = len(group)
        if size == 0:
            continue

        score = 0.0
        labels = [row[1] for row in group]

        for class_val in classes:
            p = labels.count(class_val) / size
            score += p * p

        gini_score += (1 - score) * (size / total)

    return gini_score


# ---------------------------
# Split Dataset
# ---------------------------
def split(index, value, dataset):
    left, right = [], []
    for row in dataset:
        if row[0][index] < value:
            left.append(row)
        else:
            right.append(row)
    return left, right


# ---------------------------
# Best Split
# ---------------------------
def get_best_split(dataset, n_features):
    class_values = list(set(row[1] for row in dataset))
    best_index, best_value, best_score, best_groups = None, None, float("inf"), None

    features = random.sample(range(len(dataset[0][0])), n_features)

    for index in features:
        for row in dataset:
            groups = split(index, row[0][index], dataset)
            gini_score = gini(groups, class_values)

            if gini_score < best_score:
                best_index = index
                best_value = row[0][index]
                best_score = gini_score
                best_groups = groups

    return {
        'index': best_index,
        'value': best_value,
        'groups': best_groups
    }


# ---------------------------
# Leaf Node
# ---------------------------
def to_leaf(group):
    outcomes = [row[1] for row in group]
    return Counter(outcomes).most_common(1)[0][0]


# ---------------------------
# Build Tree
# ---------------------------
def build_tree(dataset, max_depth, min_size, n_features, depth=0):
    node = get_best_split(dataset, n_features)
    left, right = node['groups']

    # If no split
    if not left or not right:
        return to_leaf(left + right)

    # Max depth reached
    if depth >= max_depth:
        return {
            'index': node['index'],
            'value': node['value'],
            'left': to_leaf(left),
            'right': to_leaf(right)
        }

    # If small group
    if len(left) <= min_size:
        left_node = to_leaf(left)
    else:
        left_node = build_tree(left, max_depth, min_size, n_features, depth + 1)

    if len(right) <= min_size:
        right_node = to_leaf(right)
    else:
        right_node = build_tree(right, max_depth, min_size, n_features, depth + 1)

    return {
        'index': node['index'],
        'value': node['value'],
        'left': left_node,
        'right': right_node
    }


# ---------------------------
# Predict (Tree)
# ---------------------------
def predict_tree(node, row):
    if not isinstance(node, dict):
        return node

    if row[node['index']] < node['value']:
        return predict_tree(node['left'], row)
    else:
        return predict_tree(node['right'], row)


# ---------------------------
# Bootstrap Sampling
# ---------------------------
def bootstrap_sample(dataset):
    return [random.choice(dataset) for _ in range(len(dataset))]


# ---------------------------
# Random Forest
# ---------------------------
def random_forest(train, test, n_trees, max_depth, min_size, n_features):
    trees = []

    for _ in range(n_trees):
        sample = bootstrap_sample(train)
        tree = build_tree(sample, max_depth, min_size, n_features)
        trees.append(tree)

    predictions = []
    for row in test:
        votes = [predict_tree(tree, row[0]) for tree in trees]
        predictions.append(Counter(votes).most_common(1)[0][0])

    return predictions


# ---------------------------
# Accuracy
# ---------------------------
def accuracy(actual, predicted):
    correct = sum(1 for i in range(len(actual)) if actual[i] == predicted[i])
    return correct / len(actual) * 100


# ---------------------------
# Main
# ---------------------------
dataset = load_data()
train, test = train_test_split(dataset)

actual = [row[1] for row in test]

# Default
predictions = random_forest(train, test, n_trees=10, max_depth=5, min_size=2, n_features=2)
acc = accuracy(actual, predictions)

print("Accuracy with 10 trees:", acc)

# Hyperparameter tuning
best_acc = 0
best_trees = 0

for trees in [5, 10, 20, 50]:
    preds = random_forest(train, test, trees, max_depth=5, min_size=2, n_features=2)
    acc = accuracy(actual, preds)

    print(f"Trees: {trees}, Accuracy: {acc}")

    if acc > best_acc:
        best_acc = acc
        best_trees = trees

print("\nBest Accuracy:", best_acc)
print("Best Number of Trees:", best_trees)

Accuracy with 10 trees: 90.0
Trees: 5, Accuracy: 93.33333333333333
Trees: 10, Accuracy: 93.33333333333333
Trees: 20, Accuracy: 90.0
Trees: 50, Accuracy: 93.33333333333333

Best Accuracy: 93.33333333333333
Best Number of Trees: 5
